<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/plot_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Metrics Comparison

Load `0_metrics.json` from multiple training runs and compare selected metrics.

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── Configure your runs here ──────────────────────────────────────────────────
# Each entry: (label, path to the training folder containing 0_metrics.json)

RUNS: list[tuple[str, str]] = [
    # ("run-1 (B=1k)", "/content/drive/MyDrive/models/exp_20260223_16-14/"),
    # ("run-2 (B=10k)", "/content/drive/MyDrive/models/exp_20260223_16-16/"),
    # ("run-3 (B=100k)", "/content/drive/MyDrive/models/exp_20260223_16-22/"),
    # ("run-4 (B=100k) #2", "/content/drive/MyDrive/models/exp_20260223_16-41/"),
    # ("run-5 (B=100)", "/content/drive/MyDrive/models/exp_20260223_16-58/"),
    # ("run-6 (B=1k) lr", "/content/drive/MyDrive/models/exp_20260223_23-44/"),
    # ("run-7 (B=10k) lr", "/content/drive/MyDrive/models/exp_20260223_23-48/"),
    # ("run-8 (B=10k) target", "/content/drive/MyDrive/models/exp_20260223_23-53/"),
    # ("run-09 target baseline", "/content/drive/MyDrive/models/exp_20260224_12-26/"),
    # ("run-10 target-net action", "/content/drive/MyDrive/models/exp_20260224_12-31/"),
    # ("run-11 target γ=.99999", "/content/drive/MyDrive/models/exp_20260224_12-36/"),
    # ("run-12 target γ=.99999 B=20k", "/content/drive/MyDrive/models/exp_20260224_21-49/"),
    # ("run-13 target γ=.99999 150x8","/content/drive/MyDrive/models/exp_20260225_01-32/",),
    # ("run-14 target γ=.99999 target-net act", "/content/drive/MyDrive/models/exp_20260225_01-37/"),
    # ("run-15 target const lr=1e-4", "/content/drive/MyDrive/models/exp_20260225_01-41/"),
    # ("run-16 lr=3e-4=const.", "/content/drive/MyDrive/models/exp_20260225_09-14/"),
    # ("run-17 B=5000", "/content/drive/MyDrive/models/exp_20260225_10-24/"),
    # ("run-18 elig_traces-base", "/content/drive/MyDrive/models/exp_20260225_10-56/"),
    # ("run-19 elig_traces lam=.8", "/content/drive/MyDrive/models/exp_20260225_21-54/"),
    # ("run-20 elig_traces tr=10", "/content/drive/MyDrive/models/exp_20260225_21-59/"),
    # ("run-21 elig_traces, lr=3e-5=const","/content/drive/MyDrive/models/exp_20260225_22-02/"),
    # ("run-22 elig_traces (=base)", "/content/drive/MyDrive/models/exp_20260226_09-41/"),
    # ("run-23 elig_traces lam=0.6 tr=10","/content/drive/MyDrive/models/exp_20260226_09-42/"),
    # ("run-24 elig_traces lam=0.6","/content/drive/MyDrive/models/exp_20260226_09-43/"),
    # ("run-25 elig_traces tr=16","/content/drive/MyDrive/models/exp_20260226_15-33/"),
    # ("run-26 elig_traces tau/2", "/content/drive/MyDrive/models/exp_20260226_15-34/"),
    # ("run-27 elig_traces 2tau", "/content/drive/MyDrive/models/exp_20260226_15-35/"),
    # ("run-28 elig_traces 8tau", "/content/drive/MyDrive/models/exp_20260226_21-06/"),
    # ("run-29 elig_traces 4tau", "/content/drive/MyDrive/models/exp_20260226_21-08/"),
    # ("run-30 elig_traces tr=32","/content/drive/MyDrive/models/exp_20260226_21-09/"),
    ("lam=0.7", "/content/drive/MyDrive/models/exp_20260322_19-54/"),
    ("lam=0.8", "/content/drive/MyDrive/models/exp_20260322_19-59/"),
    # ("lam=0.6", "/content/drive/MyDrive/models/exp_20260322_20-03/"), # -> slightly slower learning
    # ("lam=0.9", "/content/drive/MyDrive/models/exp_20260324_03-52/"), # -> optimum sumwhere between 0.7 and 0.9
    # ("lr=0.0005", "/content/drive/MyDrive/models/exp_20260324_03-53/"), # -> no clear benefit
    # ("lr=0.001", "/content/drive/MyDrive/models/exp_20260324_03-54/"), # -> already too high
]

# ── Global plot settings ──────────────────────────────────────────────────────
# X-axis mode: "step" for training steps, "time" for elapsed wallclock time (hours)
X_AXIS: str = "step"  # "step" or "time"

# Default axis limits (None = auto). Override per-plot via xlim/ylim kwargs.
XLIM: tuple[float, float] | None = None
YLIM: tuple[float, float] | None = None

# ── Load (with automatic repeat detection) ────────────────────────────────────
# runs_all[label] = list of repeats, each repeat is a list of metric dicts
# For single-run experiments (n_repeats=1 or legacy), this is a list of length 1.
runs_all: dict[str, list[list[dict]]] = {}
# runs[label] = first repeat's data (backward compatible with existing code)
runs: dict[str, list[dict]] = {}

for label, folder in RUNS:
    folder_path = Path(folder)
    repeat_dirs = sorted(folder_path.glob("repeat_*"))

    if repeat_dirs:
        # Multi-repeat experiment (n_repeats > 1)
        repeats = []
        for rd in repeat_dirs:
            p = rd / "0_metrics.json"
            if not p.exists():
                print(f"WARNING: {p} not found, skipping repeat dir '{rd.name}'")
                continue
            with p.open() as f:
                repeats.append(json.load(f))
        if not repeats:
            print(f"WARNING: No valid repeats in '{folder}', skipping '{label}'")
            continue
        runs_all[label] = repeats
        runs[label] = repeats[0]
        print(
            f"Loaded '{label}': {len(repeats)} repeats, "
            f"{len(repeats[0])} entries each from {folder_path}"
        )
    else:
        # Single-run (n_repeats=1 or legacy) — flat structure
        p = folder_path / "0_metrics.json"
        if not p.exists():
            print(f"WARNING: {p} not found, skipping '{label}'")
            continue
        with p.open() as f:
            data = json.load(f)
        runs_all[label] = [data]
        runs[label] = data
        print(f"Loaded '{label}': {len(data)} entries from {p}")

print(f"\n{len(runs)} run(s) loaded.")

## Training Parameters Comparison

Load `0_params.json` from each run and display a side-by-side comparison table.

In [ ]:
# ── Load and display training parameters ──────────────────────────────────────
import pandas as pd

# from techdays26.utils import extract_params_from_log

run_params: dict[str, dict] = {}
for label, folder in RUNS:
    p = Path(folder) / "0_params.json"
    if not p.exists():
        # Fallback: extract from 0_log.txt for older runs
        log_path = Path(folder) / "0_log.txt"
        if log_path.exists():
            print(f"'{label}': no 0_params.json, extracting from 0_log.txt...")
            extract_params_from_log(folder)
            p = Path(folder) / "0_params.json"
        else:
            print(
                f"WARNING: no 0_params.json or 0_log.txt in '{folder}', skipping '{label}'"
            )
            continue
    with p.open() as f:
        run_params[label] = json.load(f)

if run_params:
    df = pd.DataFrame(run_params).T  # rows = runs, columns = params
    df.index.name = "run"
    # Show only the most interesting columns (skip version strings etc.)
    display_cols = [
        c
        for c in df.columns
        if c
        not in (
            "commit_sha",
            "python_version",
            "pytorch_version",
            "techdays26_version",
            "ntuple_hash",
        )
    ]
    display(df[display_cols].T)
else:
    print("No 0_params.json files found.")

In [ ]:
# ── Helper: extract a metric as numpy arrays ─────────────────────────────────


def _normalized_auc(x: np.ndarray, y: np.ndarray) -> float | None:
    """Compute normalized AUC (= mean value) via trapezoidal rule.

    Returns AUC / (x_max - x_min), i.e. the average y-value over the x-range.
    Returns None if fewer than 2 finite points.
    """
    mask = np.isfinite(x) & np.isfinite(y)
    xf, yf = x[mask], y[mask]
    if len(xf) < 2:
        return None
    x_range = xf[-1] - xf[0]
    if x_range == 0:
        return None
    return float(np.trapezoid(yf, xf) / x_range)


def _format_nauc(val: float | None) -> str:
    """Format a normalized AUC value for display in a legend label."""
    if val is None:
        return "n/a"
    return f"{val:.4g}"


def _resolve_x_axis(x_axis: str | None) -> str:
    """Return the effective x-axis mode: per-plot override or global default."""
    return x_axis if x_axis is not None else X_AXIS


def get_x_axis(data: list[dict], x_axis: str | None = None) -> tuple[np.ndarray, str]:
    """Return (x_values, x_label) based on x-axis mode.

    - "step": uses the 'step' field directly.
    - "time": uses 'training_elapsed_s' converted to hours.
    """
    mode = _resolve_x_axis(x_axis)
    if mode == "time":
        x = (
            np.array([m.get("training_elapsed_s", 0.0) for m in data], dtype=np.float64)
            / 3600.0
        )
        return x, "elapsed time (h)"
    else:
        x = np.array([m["step"] for m in data])
        return x, "step"


def get_metric(
    data: list[dict], key: str, x_axis: str | None = None
) -> tuple[np.ndarray, np.ndarray, str]:
    """Return (x, y, x_label) arrays for the given metric key."""
    x, x_label = get_x_axis(data, x_axis)
    y = np.array([m[key] for m in data], dtype=np.float64)
    return x, y, x_label


# ── Aggregation utilities for repeated experiments ────────────────────────────


def get_metric_with_stats(
    label: str, key: str, x_axis: str | None = None
) -> tuple[np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Return (x, y_mean, y_std, x_label, n_repeats) across all repeats for a label.

    For single-repeat data, y_std is all zeros. Steps are aligned by index
    (all repeats log at the same step numbers).
    """
    repeats = runs_all[label]
    n_rep = len(repeats)

    # Use first repeat for x-axis (steps are identical across repeats)
    x, x_label = get_x_axis(repeats[0], x_axis)

    # Truncate to shortest repeat length
    min_len = min(len(r) for r in repeats)
    x = x[:min_len]

    ys = np.array(
        [[m[key] for m in r[:min_len]] for r in repeats], dtype=np.float64
    )  # shape: (n_repeats, n_steps)

    y_mean = ys.mean(axis=0)
    y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)

    return x, y_mean, y_std, x_label, n_rep


def bootstrap_ci(
    values: np.ndarray, n_boot: int = 10000, ci: float = 0.95
) -> tuple[float, float]:
    """Compute bootstrap confidence interval for a 1D array of values.

    Args:
        values: 1D array of observed values (e.g., from n repeats at one step).
        n_boot: Number of bootstrap resamples.
        ci: Confidence level (e.g., 0.95 for 95% CI).

    Returns:
        (ci_low, ci_high) percentile-based confidence interval for the mean.
    """
    n = len(values)
    if n < 2:
        return float(values[0]), float(values[0])
    rng = np.random.default_rng(42)
    boot_means = np.array([
        rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    return float(np.percentile(boot_means, 100 * alpha)), float(
        np.percentile(boot_means, 100 * (1 - alpha))
    )


def bootstrap_ci_array(
    ys: np.ndarray, n_boot: int = 10000, ci: float = 0.95
) -> tuple[np.ndarray, np.ndarray]:
    """Compute bootstrap CIs across all steps.

    Args:
        ys: shape (n_repeats, n_steps).

    Returns:
        (ci_low, ci_high) arrays of shape (n_steps,).
    """
    n_steps = ys.shape[1]
    ci_low = np.empty(n_steps)
    ci_high = np.empty(n_steps)
    for i in range(n_steps):
        lo, hi = bootstrap_ci(ys[:, i], n_boot=n_boot, ci=ci)
        ci_low[i] = lo
        ci_high[i] = hi
    return ci_low, ci_high


def _apply_limits(ax, xlim, ylim):
    """Apply xlim/ylim to an axis, falling back to the global defaults."""
    xl = xlim if xlim is not None else XLIM
    yl = ylim if ylim is not None else YLIM
    if xl is not None:
        ax.set_xlim(*xl)
    if yl is not None:
        ax.set_ylim(*yl)


def plot_metric(
    metric_key: str,
    *,
    x_axis: str | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlabel: str | None = None,
    yscale: str = "linear",
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (10, 4),
    filter_inf: bool = True,
    use_bootstrap: bool = False,
):
    """Plot a single metric across all loaded runs.

    For multi-repeat experiments, shows mean line with std shading (or bootstrap CI).
    """
    fig, ax = plt.subplots(figsize=figsize)
    x_label_auto = "step"
    for label in runs_all:
        x, y_mean, y_std, x_label_auto, n_rep = get_metric_with_stats(
            label, metric_key, x_axis
        )
        if filter_inf:
            mask = np.isfinite(y_mean)
            x, y_mean, y_std = x[mask], y_mean[mask], y_std[mask]
        nauc = _normalized_auc(x, y_mean)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""
        (line,) = ax.plot(
            x,
            y_mean,
            marker=".",
            markersize=3,
            linewidth=1,
            label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
        )
        if n_rep > 1:
            color = line.get_color()
            if use_bootstrap:
                repeats = runs_all[label]
                min_len = min(len(r) for r in repeats)
                ys = np.array(
                    [[m[metric_key] for m in r[:min_len]] for r in repeats],
                    dtype=np.float64,
                )
                if filter_inf:
                    ys = ys[:, mask[:min_len] if len(mask) >= min_len else mask]
                ci_lo, ci_hi = bootstrap_ci_array(ys)
                ax.fill_between(x, ci_lo, ci_hi, alpha=0.2, color=color)
            else:
                ax.fill_between(
                    x, y_mean - y_std, y_mean + y_std, alpha=0.2, color=color
                )
    ax.set_xlabel(xlabel or x_label_auto)
    ax.set_ylabel(ylabel or metric_key)
    ax.set_title(title or metric_key)
    ax.set_yscale(yscale)
    _apply_limits(ax, xlim, ylim)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_metrics_grid(
    metric_keys: list[str],
    *,
    x_axis: str | None = None,
    ncols: int = 2,
    figsize_per_plot: tuple[float, float] = (6, 3.5),
    yscales: dict[str, str] | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    filter_inf: bool = True,
    use_bootstrap: bool = False,
):
    """Plot multiple metrics in a grid of subplots.

    For multi-repeat experiments, shows mean line with std shading (or bootstrap CI).
    """
    yscales = yscales or {}
    n = len(metric_keys)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
        squeeze=False,
    )
    for idx, key in enumerate(metric_keys):
        ax = axes[idx // ncols][idx % ncols]
        x_label_auto = "step"
        for label in runs_all:
            x, y_mean, y_std, x_label_auto, n_rep = get_metric_with_stats(
                label, key, x_axis
            )
            if filter_inf:
                mask = np.isfinite(y_mean)
                x, y_mean, y_std = x[mask], y_mean[mask], y_std[mask]
            nauc = _normalized_auc(x, y_mean)
            rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""
            (line,) = ax.plot(
                x,
                y_mean,
                marker=".",
                markersize=2,
                linewidth=1,
                label=f"{label}{rep_suffix} [{_format_nauc(nauc)}]",
            )
            if n_rep > 1:
                color = line.get_color()
                if use_bootstrap:
                    repeats = runs_all[label]
                    min_len = min(len(r) for r in repeats)
                    ys = np.array(
                        [[m[key] for m in r[:min_len]] for r in repeats],
                        dtype=np.float64,
                    )
                    if filter_inf:
                        ys = ys[:, mask[:min_len] if len(mask) >= min_len else mask]
                    ci_lo, ci_hi = bootstrap_ci_array(ys)
                    ax.fill_between(x, ci_lo, ci_hi, alpha=0.2, color=color)
                else:
                    ax.fill_between(
                        x, y_mean - y_std, y_mean + y_std, alpha=0.2, color=color
                    )
        ax.set_title(key)
        ax.set_xlabel(x_label_auto)
        ax.set_yscale(yscales.get(key, "linear"))
        _apply_limits(ax, xlim, ylim)
        ax.legend(fontsize="small")
        ax.grid(True, alpha=0.3)
    # Hide unused subplots
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)
    plt.tight_layout()
    plt.show()

## Loss

In [ ]:
plot_metric("loss", title="Training Loss (MSE)", ylabel="loss")

## Relative Weight Update ||dW|| / ||W||

In [ ]:
plot_metric(
    "rel_weight_update",
    title="Relative Weight Update  ||\u0394W|| / ||W||",
    ylabel="||\u0394W|| / ||W||",
    yscale="log",
)

## Value Function Statistics

In [ ]:
plot_metrics_grid(
    ["V_old_mean", "V_old_std", "V_old_abs_mean", "V_old_min", "V_old_max"],
    ncols=2,
)

## Weight and Gradient Norms

In [ ]:
plot_metrics_grid(
    ["W_norm", "delta_W_norm", "grad_mean", "grad_std"],
    ncols=2,
)

## Training Dynamics

In [ ]:
plot_metrics_grid(
    ["done_frac", "n_wins", "moves_left_mean", "lr"],
    ncols=2,
)

## Wall Time per Logging Interval

In [ ]:
plot_metric("wall_time_s", title="Wall Time per Interval", ylabel="seconds")

## Custom: Pick Any Metric

Available keys in each metrics entry:

In [ ]:
# Print all available metric keys
sample_run = next(iter(runs.values()))
print("Available metrics:")
for k in sample_run[0].keys():
    print(f"  - {k}")

In [ ]:
# Plot any metric by name:
# plot_metric("grad_nnz", title="Non-zero Gradient Entries")

---

# Arena Results Comparison

Load `0_arena_metrics.json` from each run and plot pre-computed scores over training steps.
Falls back to individual `step_X_arena_result.json` files for older runs.

In [ ]:
# ── Load arena results for each run from 0_arena_metrics.json ────────────────


def load_arena_results(folder: str) -> dict[int, list[dict]]:
    """Load arena aggregate scores from 0_arena_metrics.json.

    Falls back to reading individual step_X_arena_result.json files
    if the compact file doesn't exist (for older runs).

    Returns: {step: list_of_aggregate_rows}
    """
    folder = Path(folder)

    # Preferred: single compact file
    compact = folder / "0_arena_metrics.json"
    if compact.exists():
        with compact.open() as f:
            data = json.load(f)
        return {entry["step"]: entry["aggregates"] for entry in data}

    # Fallback: individual files (slow, for older runs)
    import re

    results = {}
    for p in sorted(folder.glob("step_*_arena_result.json")):
        m = re.search(r"step_(\d+)_arena_result", p.name)
        if m is None:
            continue
        step = int(m.group(1))
        with p.open() as f:
            data = json.load(f)
        results[step] = data["result"]["aggregates"]
    return dict(sorted(results.items()))


def load_arena_results_all_repeats(
    folder: str,
) -> list[dict[int, list[dict]]]:
    """Load arena results for all repeats (or single run).

    Returns a list of per-repeat arena dicts.
    For single-run experiments, returns a list of length 1.
    """
    folder = Path(folder)
    repeat_dirs = sorted(folder.glob("repeat_*"))

    if repeat_dirs:
        return [load_arena_results(str(rd)) for rd in repeat_dirs]
    else:
        return [load_arena_results(str(folder))]


# Load arena results for all configured runs
arena_runs: dict[str, dict[int, list[dict]]] = {}
# arena_runs_all[label] = list of per-repeat arena dicts
arena_runs_all: dict[str, list[dict[int, list[dict]]]] = {}

for label, folder in RUNS:
    all_repeats = load_arena_results_all_repeats(folder)
    # Filter out empty repeats
    all_repeats = [r for r in all_repeats if r]
    if not all_repeats:
        print(f"WARNING: No arena result files found in '{folder}', skipping '{label}'")
        continue
    arena_runs_all[label] = all_repeats
    arena_runs[label] = all_repeats[0]  # backward compatible: first repeat
    steps = sorted(all_repeats[0].keys())
    n_rep = len(all_repeats)
    rep_info = f" ({n_rep} repeats)" if n_rep > 1 else ""
    print(
        f"Loaded '{label}': {len(all_repeats[0])} arena snapshots at steps {steps}{rep_info}"
    )

print(f"\n{len(arena_runs)} run(s) with arena results.")

In [ ]:
# ── Arena helpers ─────────────────────────────────────────────────────────────


def get_matchup_key(row: dict) -> str:
    """Matchup identifier: 'agent_yellow vs agent_red'."""
    return f"{row['agent_yellow']} vs {row['agent_red']}"


def list_matchups(arena_data: dict[int, list[dict]]) -> list[str]:
    """List all unique matchup keys across all steps (epsilon=0 only)."""
    matchups = set()
    for rows in arena_data.values():
        for r in rows:
            if r["epsilon_yellow"] == 0.0 and r["epsilon_red"] == 0.0:
                matchups.add(get_matchup_key(r))
    return sorted(matchups)


def _step_to_time_map(label: str) -> dict[int, float]:
    """Build a mapping from step -> elapsed time (hours) using training_elapsed_s."""
    if label not in runs:
        return {}
    data = runs[label]
    return {int(m["step"]): m.get("training_elapsed_s", 0.0) / 3600.0 for m in data}


def _map_steps_to_x(
    steps: list[int], label: str, x_axis: str | None = None
) -> tuple[list, str]:
    """Convert step list to x-values based on x-axis mode."""
    mode = _resolve_x_axis(x_axis)
    if mode == "time":
        mapping = _step_to_time_map(label)
        if mapping:
            known_steps = sorted(mapping.keys())
            known_times = [mapping[s] for s in known_steps]
            x = list(np.interp(steps, known_steps, known_times))
        else:
            x = [float(s) for s in steps]
        return x, "elapsed time (h)"
    return steps, "step"


def extract_series(
    arena_data: dict[int, list[dict]],
    matchup: str,
) -> dict[str, list]:
    """Extract time series for a specific matchup directly from the JSON data."""
    series: dict[str, list] = {"steps": []}

    for step, rows in sorted(arena_data.items()):
        for r in rows:
            if r["epsilon_yellow"] != 0.0 or r["epsilon_red"] != 0.0:
                continue
            if get_matchup_key(r) != matchup:
                continue
            series["steps"].append(step)
            for k in (
                "games",
                "yellow_wins",
                "red_wins",
                "draws",
                "score",
                "avg",
                "timeouts",
                "illegal_moves",
                "exceptions",
                "total_time_s",
            ):
                series.setdefault(k, []).append(r.get(k, 0))
            break  # one row per matchup per step

    return series


def extract_series_with_stats(
    label: str,
    matchup: str,
    metric: str,
) -> tuple[list[int], np.ndarray, np.ndarray, int]:
    """Extract mean and std of a metric across repeats for a matchup.

    Returns: (steps, y_mean, y_std, n_repeats)
    """
    all_repeats = arena_runs_all[label]
    n_rep = len(all_repeats)

    # Get series from each repeat
    all_series = []
    ref_steps = None
    for arena_data in all_repeats:
        s = extract_series(arena_data, matchup)
        if not s["steps"]:
            continue
        all_series.append(s)
        if ref_steps is None:
            ref_steps = s["steps"]

    if not all_series:
        return [], np.array([]), np.array([]), 0

    # Truncate to common steps (shortest repeat)
    min_len = min(len(s[metric]) for s in all_series)
    steps = ref_steps[:min_len]

    ys = np.array([s[metric][:min_len] for s in all_series], dtype=np.float64)
    y_mean = ys.mean(axis=0)
    y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)

    return steps, y_mean, y_std, len(all_series)


def plot_arena_single_run(
    run_label: str,
    metric: str = "avg",
    *,
    x_axis: str | None = None,
    matchups: list[str] | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 5),
    use_bootstrap: bool = False,
):
    """Plot a metric for a single run: one line per matchup.

    For multi-repeat experiments, shows mean with error bars.
    """
    if run_label not in arena_runs:
        print(f"Run '{run_label}' not found in arena_runs")
        return
    if matchups is None:
        matchups = list_matchups(arena_runs[run_label])

    n_rep = len(arena_runs_all[run_label])
    fig, ax = plt.subplots(figsize=figsize)
    x_label = "step"
    for m in matchups:
        steps, y_mean, y_std, n_actual = extract_series_with_stats(run_label, m, metric)
        if not steps:
            continue
        x, x_label = _map_steps_to_x(steps, run_label, x_axis)
        xa, ya = np.asarray(x, dtype=np.float64), y_mean
        nauc = _normalized_auc(xa, ya)
        rep_suffix = f" (n={n_actual})" if n_actual > 1 else ""

        if n_actual > 1:
            if use_bootstrap:
                all_repeats = arena_runs_all[run_label]
                min_len = len(steps)
                ys = np.array(
                    [
                        extract_series(rd, m)[metric][:min_len]
                        for rd in all_repeats
                        if extract_series(rd, m)["steps"]
                    ],
                    dtype=np.float64,
                )
                ci_lo, ci_hi = bootstrap_ci_array(ys)
                line = ax.errorbar(
                    xa,
                    ya,
                    yerr=[ya - ci_lo, ci_hi - ya],
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    capsize=3,
                    label=f"{m}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
            else:
                line = ax.errorbar(
                    xa,
                    ya,
                    yerr=y_std,
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    capsize=3,
                    label=f"{m}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
        else:
            ax.plot(
                xa,
                ya,
                marker="o",
                markersize=4,
                linewidth=1.5,
                label=f"{m} [nAUC={_format_nauc(nauc)}]",
            )

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"[{run_label}] {metric}")
    _apply_limits(ax, xlim, ylim)
    ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
    ax.legend(fontsize="small", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_compare_runs(
    matchup: str,
    metric: str = "avg",
    *,
    x_axis: str | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 5),
    use_bootstrap: bool = False,
):
    """Compare a metric for a specific matchup across runs: one line per run.

    For multi-repeat experiments, shows mean with error bars.
    """
    fig, ax = plt.subplots(figsize=figsize)

    x_label = "step"
    for label in arena_runs_all:
        steps, y_mean, y_std, n_rep = extract_series_with_stats(label, matchup, metric)
        if not steps:
            continue
        x, x_label = _map_steps_to_x(steps, label, x_axis)
        xa, ya = np.asarray(x, dtype=np.float64), y_mean
        nauc = _normalized_auc(xa, ya)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""

        if n_rep > 1:
            if use_bootstrap:
                all_repeats = arena_runs_all[label]
                min_len = len(steps)
                ys = np.array(
                    [
                        extract_series(rd, matchup)[metric][:min_len]
                        for rd in all_repeats
                        if extract_series(rd, matchup)["steps"]
                    ],
                    dtype=np.float64,
                )
                ci_lo, ci_hi = bootstrap_ci_array(ys)
                ax.errorbar(
                    xa,
                    ya,
                    yerr=[ya - ci_lo, ci_hi - ya],
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    capsize=3,
                    label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
            else:
                ax.errorbar(
                    xa,
                    ya,
                    yerr=y_std,
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    capsize=3,
                    label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
        else:
            ax.plot(
                xa,
                ya,
                marker="o",
                markersize=4,
                linewidth=1.5,
                label=f"{label} [nAUC={_format_nauc(nauc)}]",
            )

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"{matchup} — {metric}")
    _apply_limits(ax, xlim, ylim)
    ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_compare_runs_grid(
    matchups: list[str],
    metric: str = "avg",
    *,
    x_axis: str | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 4),
    use_bootstrap: bool = False,
):
    """One subplot per matchup, one line per run. With error bars for repeats."""
    n = len(matchups)
    fig, axes = plt.subplots(n, 1, figsize=(figsize[0], figsize[1] * n), squeeze=False)

    x_label = "step"
    for idx, matchup in enumerate(matchups):
        ax = axes[idx][0]
        for label in arena_runs_all:
            steps, y_mean, y_std, n_rep = extract_series_with_stats(
                label, matchup, metric
            )
            if not steps:
                continue
            x, x_label = _map_steps_to_x(steps, label, x_axis)
            xa, ya = np.asarray(x, dtype=np.float64), y_mean
            nauc = _normalized_auc(xa, ya)
            rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""

            if n_rep > 1:
                if use_bootstrap:
                    all_repeats = arena_runs_all[label]
                    min_len = len(steps)
                    ys = np.array(
                        [
                            extract_series(rd, matchup)[metric][:min_len]
                            for rd in all_repeats
                            if extract_series(rd, matchup)["steps"]
                        ],
                        dtype=np.float64,
                    )
                    ci_lo, ci_hi = bootstrap_ci_array(ys)
                    ax.errorbar(
                        xa,
                        ya,
                        yerr=[ya - ci_lo, ci_hi - ya],
                        marker="o",
                        markersize=4,
                        linewidth=1.5,
                        capsize=3,
                        label=f"{label}{rep_suffix} [{_format_nauc(nauc)}]",
                    )
                else:
                    ax.errorbar(
                        xa,
                        ya,
                        yerr=y_std,
                        marker="o",
                        markersize=4,
                        linewidth=1.5,
                        capsize=3,
                        label=f"{label}{rep_suffix} [{_format_nauc(nauc)}]",
                    )
            else:
                ax.plot(
                    xa,
                    ya,
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    label=f"{label} [{_format_nauc(nauc)}]",
                )
        ax.set_title(matchup)
        ax.set_xlabel(x_label)
        ax.set_ylabel(ylabel or metric)
        _apply_limits(ax, xlim, ylim)
        ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
        ax.legend(fontsize="small")
        ax.grid(True, alpha=0.3)

    plt.suptitle(title or f"{metric} per matchup", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
from collections import defaultdict


def plot_arena_averaged(
    *,
    starts_with: str | None = "ntuple",
    ends_with: str | None = None,
    metric: str = "avg",
    x_axis: str | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 6),
    use_bootstrap: bool = False,
):
    """Average a metric over all matching matchups and plot one line per run.

    For multi-repeat experiments, shows mean with error bars (std or bootstrap CI).

    Args:
        starts_with: Keep only matchups starting with this string (e.g. "ntuple").
        ends_with: Keep only matchups ending with this string.
        metric: Arena metric to average (default "avg").
        x_axis: "step" or "time". Defaults to global X_AXIS.
        use_bootstrap: If True, use bootstrap CI instead of std for error bars.
    """
    ref_label = list(arena_runs_all.keys())[-1]
    all_matchups = list_matchups(arena_runs_all[ref_label][0])

    # Filter matchups
    filtered_matchups = []
    for m in all_matchups:
        if starts_with and not m.startswith(starts_with):
            continue
        if ends_with and not m.endswith(ends_with):
            continue
        filtered_matchups.append(m)

    if not filtered_matchups:
        print("No matching matchups found.")
        return

    fig, ax = plt.subplots(figsize=figsize)
    x_label = "step"
    averaged_results: dict[str, np.ndarray] = {}

    for label in arena_runs_all:
        all_repeats = arena_runs_all[label]
        n_rep = len(all_repeats)

        # For each repeat, average across matchups, then compute stats across repeats
        per_repeat_averaged = []
        ref_steps = None

        for arena_data in all_repeats:
            matchup_series = []
            for m in filtered_matchups:
                s = extract_series(arena_data, m)
                if s["steps"] and metric in s:
                    matchup_series.append(s[metric])
                    if ref_steps is None:
                        ref_steps = s["steps"]

            if not matchup_series:
                continue

            # Truncate to shortest matchup series
            min_len = min(len(ms) for ms in matchup_series)
            arrs = [np.array(ms[:min_len]) for ms in matchup_series]
            per_repeat_averaged.append(np.mean(arrs, axis=0))

        if not per_repeat_averaged or ref_steps is None:
            continue

        # Truncate to shortest repeat
        min_len = min(len(a) for a in per_repeat_averaged)
        steps = ref_steps[:min_len]
        ys = np.array([a[:min_len] for a in per_repeat_averaged], dtype=np.float64)
        y_mean = ys.mean(axis=0)
        y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)

        x, x_label = _map_steps_to_x(steps, label, x_axis)
        xa = np.asarray(x, dtype=np.float64)
        nauc = _normalized_auc(xa, y_mean)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""

        if n_rep > 1:
            if use_bootstrap:
                ci_lo, ci_hi = bootstrap_ci_array(ys)
                ax.errorbar(
                    xa,
                    y_mean,
                    yerr=[y_mean - ci_lo, ci_hi - y_mean],
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    capsize=3,
                    label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
            else:
                (line,) = ax.plot(
                    xa,
                    y_mean,
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
                )
                ax.fill_between(
                    xa,
                    y_mean - y_std,
                    y_mean + y_std,
                    alpha=0.2,
                    color=line.get_color(),
                )
        else:
            ax.plot(
                xa,
                y_mean,
                marker="o",
                markersize=4,
                linewidth=1.5,
                label=f"{label} [nAUC={_format_nauc(nauc)}]",
            )

        averaged_results[label] = y_mean

    filt = []
    if starts_with:
        filt.append(f"starts_with={starts_with!r}")
    if ends_with:
        filt.append(f"ends_with={ends_with!r}")
    filter_desc = f" ({', '.join(filt)})" if filt else ""

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or f"averaged {metric}")
    ax.set_title(title or f"Averaged Arena {metric}{filter_desc}")
    _apply_limits(ax, xlim, ylim)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return averaged_results

In [ ]:
x = plot_arena_averaged(starts_with="ntuple")
for k, v in x.items():
    print(k, v.mean())

In [ ]:
# List all available matchups (from last run)
first_label = list(arena_runs.keys())[-1]
print(f"Matchups in '{first_label}':")
for m in list_matchups(arena_runs[first_label]):
    print(f"  - {m}")

## Avg Score — All Matchups (Single Run)

In [ ]:
plot_arena_single_run(first_label, "avg")

## Compare Runs — Avg Score per Matchup

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "avg")
plot_arena_compare_runs("ntuple vs bitbully-16ply-book12ply", "avg")
plot_arena_compare_runs("bitbully-16ply-book12ply vs ntuple", "avg")

## Compare Runs — Avg Score Grid

In [ ]:
# All matchups in one grid (auto-detected from last run)
plot_arena_compare_runs_grid(
    list_matchups(arena_runs[first_label]), "avg", ylim=(-1.05, 1.05)
)

## Compare Runs — Yellow Wins

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "yellow_wins")
plot_arena_compare_runs("bitbully-full-strength vs ntuple", "yellow_wins")

## Compare Runs — Score

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "score")
plot_arena_compare_runs("bitbully-full-strength vs ntuple", "score")

## Custom Arena Plot

Use `plot_arena_compare_runs(matchup, metric)` with any matchup from the list above and any metric:
`avg`, `score`, `yellow_wins`, `red_wins`, `draws`, `games`, `timeouts`, `illegal_moves`, `exceptions`

In [ ]:
# plot_arena_compare_runs("ntuple vs random", "draws")